
# 02 — Text Preprocessing

## 1. Load the Cleaned Dataset

In this notebook, we will explore text preprocessing
techniques before training our classification model.

We will use the cleaned CFPB complaints dataset.


In [ ]:

import pandas as pd

df = pd.read_csv("../data/processed/cleaned_complaints.csv")

print("Dataset shape:", df.shape)
df[["Consumer complaint narrative", "Product"]].head()


In [ ]:

text = df["Consumer complaint narrative"].iloc[0]

print("Original complaint:\n")
print(text)


In [ ]:

from collections import Counter
import re

sample_texts = df["Consumer complaint narrative"].head(100)

words = []

for text in sample_texts:
    tokens = re.findall(r"\b\w+\b", text.lower())
    words.extend(tokens)

word_counts = Counter(words)

print(word_counts.most_common(20))


## 2. Understanding TF-IDF

TF-IDF converts text into numerical vectors based on
term frequency and inverse document frequency.

We start with a small example before applying it
to the real dataset.

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

documents = [
    "bank bank payment",
    "bank transfer",
    "loan payment"
]

vectorizer = TfidfVectorizer()

X_tfidf = vectorizer.fit_transform(documents)

tfidf_table = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print("TF-IDF Matrix Shape:", X_tfidf.shape)

tfidf_table.round(3)



## 3. Exploring TF-IDF on Real Complaints

We use 100 real complaints to explore:

- Vocabulary size
- TF-IDF matrix dimensions
- Non-zero values
- Sparse matrix representation

This is an exploratory experiment, not model training.


In [ ]:

sample = df["Consumer complaint narrative"].head(100)

vectorizer = TfidfVectorizer()

X_sample = vectorizer.fit_transform(sample)

print("Number of documents:", X_sample.shape[0])
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Matrix shape:", X_sample.shape)
print("Non-zero values:", X_sample.nnz)


In [ ]:

total_cells = X_sample.shape[0] * X_sample.shape[1]

non_zero_cells = X_sample.nnz

sparsity = (1 - non_zero_cells / total_cells) * 100

print(f"Total cells: {total_cells:,}")
print(f"Non-zero cells: {non_zero_cells:,}")
print(f"Sparsity: {sparsity:.2f}%")


In [ ]:

vectorizer_default = TfidfVectorizer()

vectorizer_filtered = TfidfVectorizer(
    min_df=2,
    max_df=0.90
)

X_default = vectorizer_default.fit_transform(sample)
X_filtered = vectorizer_filtered.fit_transform(sample)

print("Default vocabulary:", X_default.shape[1])
print("Filtered vocabulary:", X_filtered.shape[1])
print("Features removed:", X_default.shape[1] - X_filtered.shape[1])


In [ ]:

default_words = set(vectorizer_default.get_feature_names_out())

filtered_words = set(vectorizer_filtered.get_feature_names_out())

removed_words = default_words - filtered_words

print("Total removed words:", len(removed_words))
print("Examples:", sorted(removed_words)[:30])


In [ ]:

vectorizer_min = TfidfVectorizer(min_df=2)

vectorizer_max = TfidfVectorizer(max_df=0.90)

X_min = vectorizer_min.fit_transform(sample)
X_max = vectorizer_max.fit_transform(sample)

print("Default vocabulary:", X_default.shape[1])
print("With min_df=2:", X_min.shape[1])
print("With max_df=0.90:", X_max.shape[1])
print("With both:", X_filtered.shape[1])
